In [ ]:
%load_ext autoreload
%autoreload 2

from itertools import product
import sys
sys.path.append('/home/projects/nyosef/zvise/PixelGen/')
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D
from PixelGen.metrics import MultiModalVIMetrics
from sklearn.preprocessing import PowerTransformer
from pathlib import Path


import anndata as ad
import pixelator
import torch
import scvi
import scipy
# from scvi import autotune

import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

from PixelGen.pxl_utils import train_model, get_model_latents
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA
from pixelator.common.statistics import clr_transformation, dsb_normalize


from pixelator.pna.plot import molecule_rank_plot

from sklearn.preprocessing import StandardScaler, MinMaxScaler 

from PixelGen.pxl_utils import train_model, get_model_latents, convert_polarization_to_feature_matrix, \
     convert_colocalization_to_feature_matrix, download_pxl
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA, add_one_hot_encoding_obsm, plot_cumulative_variance
from PixelGen.common_utils import standardize, std_clip, filter_hv, split_pair_column, filter_df_by_two_columns, rank_plot
from PixelGen.metrics import MultiModalVIMetrics, distr_autocorrelation_in_latent
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D

import tempfile

from scvi import REGISTRY_KEYS
from scvi.module.base import (
    BaseModuleClass,
    LossOutput,
    PyroBaseModuleClass,
    auto_move_data,
)
from torch.distributions import NegativeBinomial, Normal, Poisson, MixtureSameFamily, Beta
from torch.distributions import kl_divergence as kl

print(torch.cuda.is_available())
from sklearn.decomposition import PCA

from anndata import AnnData
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
sc.set_figure_params(figsize=(6, 6), frameon=False)

sns.set_theme()
torch.set_float32_matmul_precision("high")
save_dir = tempfile.TemporaryDirectory()

from PixelGen.utils import plot_latent, plot_gene_heatmap, plot_model_latents
sys.path.insert(0, str(Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/doublet_separation')))
from doublet_separation import B_CD4_logfc_dict, B_CD8_logfc_dict, run_cellwise_coloc_analysis_to_disk, concat_abundance_to_adata, add_doublets_metadata
%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
from pixelator import read_pna as read
import pickle
import scipy.sparse as sp
from scipy.sparse.csgraph import dijkstra
import anndata
import hotspot

import numpy as np
import pandas as pd
import networkx as nx
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

import igraph as ig
import leidenalg
from joblib import Parallel, delayed
from tqdm import tqdm
import os
import numpy as np
import pandas as pd
from anndata import AnnData
import hotspot
from sqlitedict import SqliteDict

In [ ]:
# ============================================================
# CONFIG — set DATASET to choose which dataset to run
# ============================================================
DATASET = "pbmsc"  # "main" or "pbmsc"

CONFIG = {
    "main": {
        "mode": "single",   # one merged pxl dataset, cells from adata obs_names
        "annotated_adata_path": "/home/projects/nyosef/zvise/PixelGen/PixelGen/Data/adatas/final_adatas/adata_annotated.h5ad",
        "data_dir": "/home/projects/nyosef/zvise/PxlgnProject/Data",
        "cache_db": "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/HOTSPOT/hotspot_cache.sqlite",
        "final_output": "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/HOTSPOT/hotspot_final.pkl",
    },
    "pbmsc": {
        "mode": "multi",    # multiple .pxl files, keyed by sample_cellid
        "data_path": "/home/projects/nyosef/zvise/PixelGen/PixelGen/Data/new_pbmsc",
        "cache_db": "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/HOTSPOT/PBMSC_hotspot_cache.sqlite",
        "final_output": "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/HOTSPOT/PBMSC_hotspot_final.pkl",
    },
}

cfg = CONFIG[DATASET]
CACHE_DB = cfg["cache_db"]
FINAL_OUTPUT = cfg["final_output"]

# Load files / adata based on mode
if cfg["mode"] == "single":
    DATA_DIR = Path(cfg["data_dir"])
    files = [f for f in DATA_DIR.rglob('*.pxl') if f.is_file()]
    pg_data = read(files)
    adata = sc.read_h5ad(cfg["annotated_adata_path"])
elif cfg["mode"] == "multi":
    files_to_process = [f for f in Path(cfg["data_path"]).rglob('*.pxl') if f.is_file()]
    print(f"Found {len(files_to_process)} .pxl files")

In [ ]:
def run_hotspot(layout, top_n=70):
    """Run Hotspot spatial autocorrelation on a single cell layout DataFrame."""
    exclude = {'index', 'pixel_type', 'x', 'y', 'z', 'component', 'graph_projection', 'layout', 'sample'}
    marker_cols = [c for c in layout.columns if c not in exclude]
    raw_data = layout[marker_cols].values.astype(float)

    # Proportion normalization: (X - p) / sqrt(p(1-p))
    p = raw_data.sum(axis=0) / raw_data.sum()
    denom = np.sqrt(p * (1 - p))
    valid = denom > 0
    standardized = ((raw_data[:, valid] - p[valid]) / denom[valid]) + 1e-6
    markers = np.array(marker_cols)[valid]

    adata = AnnData(standardized, var=pd.DataFrame(index=markers))
    adata.obsm["spatial"] = layout[["x", "y", "z"]].values

    hs = hotspot.Hotspot(adata, model='none', latent_obsm_key="spatial")
    hs.create_knn_graph(n_neighbors=30)
    pol = hs.compute_autocorrelations(jobs=-1)
    top_markers = pol.nlargest(top_n, 'Z').index.tolist()
    hs.compute_local_correlations(genes=top_markers, jobs=-1)

    mat = hs.local_correlation_c
    np.fill_diagonal(mat.values, pol.loc[mat.index, 'C'])
    iu = np.triu_indices(len(mat))
    flat_df = pd.DataFrame(
        [mat.values[iu]],
        columns=[f"{mat.index[i]}/{mat.columns[j]}" for i, j in zip(*iu)],
        index=[layout['component'].iloc[0]]
    )
    return flat_df


def run_hotspot_single(cell_ids, pg_data, cache_path):
    """Process a single merged pxl dataset, using cell_ids from adata.obs_names."""
    with SqliteDict(cache_path, autocommit=True) as cache:
        todo = [cid for cid in cell_ids if cid not in cache]
        print(f"Total: {len(cell_ids)} | Cached: {len(cache)} | Remaining: {len(todo)}")

        for cid in tqdm(todo, desc="Processing cells"):
            try:
                layout_df = pg_data.filter(components=cid).precomputed_layouts().to_df()
                if layout_df.empty:
                    continue
                result_row = run_hotspot(layout_df)
                cache[cid] = result_row.iloc[0]
            except Exception as e:
                print(f"⚠️ Error on {cid}: {e}")
                continue

        print("Reassembling final dataset...")
        final_df = pd.DataFrame.from_dict(dict(cache), orient="index").fillna(0)
    return final_df


def run_hotspot_multifile(file_paths, cache_path):
    """Process multiple .pxl files, keying results as sample_cellid."""
    with SqliteDict(cache_path, autocommit=True) as cache:
        for fpath in file_paths:
            fpath = Path(fpath)
            sample_name = fpath.stem.split('.')[0]
            print(f"\n--- Loading: {fpath.name} ---")
            try:
                pg_data = read(fpath)
                adata = pg_data.adata()
                todo_map = {
                    f"{sample_name}_{cid}": cid
                    for cid in adata.obs_names
                    if f"{sample_name}_{cid}" not in cache
                }
                print(f"Total: {len(adata.obs_names)} | Cached: {len(adata.obs_names) - len(todo_map)} | To process: {len(todo_map)}")

                for unique_key, cid in tqdm(todo_map.items(), desc=f"Processing {sample_name}"):
                    try:
                        layout_df = pg_data.filter(components=cid).precomputed_layouts().to_df()
                        if layout_df.empty:
                            continue
                        result_row = run_hotspot(layout_df)
                        result_row.index = [unique_key]
                        cache[unique_key] = result_row.iloc[0]
                    except Exception as e:
                        print(f"⚠️ Error on {unique_key}: {e}")
                        continue
            except Exception as e:
                print(f"❌ Failed to load {fpath.name}: {e}")
                continue

        print("\nReassembling final dataset from all files...")
        final_df = pd.DataFrame.from_dict(dict(cache), orient="index").fillna(0)
    return final_df

In [ ]:
# Run hotspot based on configured mode
if cfg["mode"] == "single":
    cell_ids = list(adata.obs_names)
    results = run_hotspot_single(cell_ids, pg_data, CACHE_DB)
elif cfg["mode"] == "multi":
    results = run_hotspot_multifile(files_to_process, CACHE_DB)

results.to_pickle(FINAL_OUTPUT)
print(f"Saved {len(results)} cells to {FINAL_OUTPUT}")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Load the results
FINAL_OUTPUT = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/HOTSPOT/hotspot_final.pkl"
df = pd.read_pickle(FINAL_OUTPUT)

# 2. Identify the Top 10 correlated pairs (by mean value across all cells)
# We exclude columns that are entirely zero if any exist
mean_corrs = df.mean().sort_values(ascending=False)
top_10_pairs = mean_corrs.head(10).index.tolist()

print("Top 10 Highly Correlated Pairs (Mean):")
for pair in top_10_pairs:
    print(f"{pair}: {mean_corrs[pair]:.4f}")

# 3. Plot Distributions
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 5, figsize=(22, 10))
axes = axes.flatten()

for i, pair in enumerate(top_10_pairs):
    # Plot histogram with KDE
    sns.histplot(df[pair], kde=True, ax=axes[i], color='teal', bins=30)
    
    # Beautify
    axes[i].set_title(f"{i+1}. {pair}", fontsize=12, fontweight='bold')
    axes[i].set_xlabel("Local Correlation ($C$)", fontsize=10)
    axes[i].set_ylabel("Frequency", fontsize=10)
    
    # Add a vertical line for the mean
    axes[i].axvline(mean_corrs[pair], color='red', linestyle='--', alpha=0.7, label=f'Mean: {mean_corrs[pair]:.3f}')
    axes[i].legend(fontsize=8)

plt.suptitle("Distributions of Top 10 Sptially Correlated Protein Pairs", fontsize=18, y=1.02)
plt.tight_layout()

# Save or show
plt.savefig("top_10_correlations.png", dpi=300, bbox_inches='tight')
plt.show()